## See "Hard Mining Negatives for Semantic Similarity"

https://www.kaggle.com/code/jithinanievarghese/hard-mining-negatives-for-semantic-similarity#Load-Data-and-preprocess-data

In [3]:
import os
import sys
PROJECT_ROOT = os.path.abspath(os.path.join(
 os.getcwd(),
 os.pardir+'/playground')
)
#only add it once
if (PROJECT_ROOT not in sys.path):
 sys.path.append(PROJECT_ROOT)

import utils as ut
from sentence_transformers import SentenceTransformer, util
from tqdm import tqdm
import numpy as np 
import pandas as pd 
import csv

/home/kperkins411/anaconda3/envs/p311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Token is valid (permission: write).
Your token has been saved in your configured git credential helpers (store).
Your token has been saved to /home/kperkins411/.cache/huggingface/token
Login successful


In [4]:
def preprocess_text(text):
    """
    clean white space and lower case the text
    """
    return " ".join(text.split()).lower()

In [13]:
df = pd.read_json('../data/trn.json')

In [15]:
df.head()

,anchor,positive,most_dissimilar_context,id
0,What safeguards are in place to protect the in...,Information We Collect From Other Sources We m...,If such Standard Cost methodology change resul...,0
1,Is there a guarantee from the manufacturers re...,Each of the Suppliers warrants that the Produc...,We do not obtain your age range and gender.,1
2,What type of authorization has the video confe...,Skype hereby grants to Online BVI and the Comp...,"In order to keep your exclusivity, you agree t...",2
3,Can the Blockchain Administrator arrange for t...,(a) The Fund hereby employs the Blockchain Adm...,8. Insurance. During the Term of this Agreemen...,3
4,What happens if a Party fails to retain record...,Each Party will retain such records for at lea...,"""Web Beacons"" (also known as Web bugs, pixel t...",4


In [16]:
# df = df.rename(columns={"Product Title": "positive", 'Cluster ID': 'cluster_id', "Cluster Label": "anchor", "Category Label": "category"})
df.reset_index(drop=True, inplace=True)
df.drop_duplicates(subset=['anchor', 'positive'], inplace=True)
df.drop_duplicates(subset=['anchor'], inplace=True)
df.drop_duplicates(subset=['positive'], inplace=True)
df.reset_index(drop=True, inplace=True)

In [17]:
df.anchor = df['anchor'].apply(lambda x: preprocess_text(x))
df.positive = df['positive'].apply(lambda x: preprocess_text(x))

In [19]:
class HardMineNegatives():
    """
    Hard-mining Negatives for training a semantic similairty task with Triplet Loss.
    Here we find the nearest negatives of a query in a search pool 
    by using sentence transformer model embeddings and cosine similarity ratio.
    param: model_path: path of sentence transformer model
    param: search_max_threshold:  maximimum cosine similarity ratio
    param: search_min_threshold: minimum cosine similarity ratio
    param: search_limit: total length of data in which we want to search, only if search pool length is very high
    param: top_n_results: number of top nearest negative  to be returned, default is 1
    """
    def __init__(self, model_path: str, **kwargs):
        self.model = SentenceTransformer(model_path)
        self.search_max_threshold = kwargs['search_max_threshold'],
        self.search_min_threshold = kwargs['search_min_threshold']
        self.search_limit = kwargs.get('search_limit')
        self.top_n_results = kwargs.get('top_n_results') if kwargs.get('top_n_results') else 1

    def get_hard_mined_negatives(self, anchor: str,  search_pool:np.ndarray):
        """
        to retrieve embeddings from sentence transformer model for anchor and sentences in search pool,
        find the cosine similairty ratio between the  anchor and search pool sentences,
        apply search thresholds and return the top nearest negatives based on the highest
        cosine similarity scores.
        if no data is found in between the self.search_max_threshold and self.search_min_threshold ratios,
        we will take the results between 0 and less than self.search_min_threshold ratios (this is an extreme case)

        param: anchor: source text to which we need to find the nearest negative
        param: search_pool: numpy array of sentences from which
               we need to find the cosine similarity ratios with the anchor.
               any meta value for sentences can be given after next index of
               sentence, in the form
               search_pool = array([
                    ['apple iphone 8 256 gb gold', "mobile", "1001"],
                    ['apple iphone 7 plus 32gb silver', "1002"]])
               where "mobile", "1001" are meta values,
               the returned results will contain the respective cosine similarity
               ratio at the last index of each sentence array
               result = array([
                    ['apple iphone 8 256 gb gold', "mobile", "1001", 69.5],
                    ['apple iphone 7 plus 32gb silver', "1002", 70.5]])
               where 69.5 and 70.5 are cosine similarity ratios.
        """
        self.search_limit = self.search_limit if self.search_limit else search_pool.shape[0]
        search_pool = search_pool[: self.search_limit]
        # shuffle data to search in random pool of data, in case of search limit less than the total length
        np.random.shuffle(search_pool)
        sentences = [anchor] + [row[0] for row in search_pool]
        embeddings = self.model.encode(sentences, convert_to_tensor=False)
        source_vector = embeddings[0]
        # calculate the cosine similairty with the other sentences in search pool
        similarity = [round(util.cos_sim(source_vector, embed).numpy()[0][0]*100, 2) for embed in embeddings[1:]]
        similarity = np.array(similarity)
        negative_indices = np.where((similarity <= self.search_max_threshold) & (similarity >= self.search_min_threshold))
        if not negative_indices[0].shape[0]:
            negative_indices = np.where((similarity < self.search_min_threshold) & (similarity >= 0))
        negative_indices = negative_indices[0]
        # take respective selected indices
        search_pool = np.take(search_pool, negative_indices, axis=0)
        similarity = np.take(similarity, negative_indices, axis=0)
        # reshape to concatenate with meta values of search pool
        similarity = similarity.reshape(-1, 1)
        # concat the ratio to the meta values of search pool
        search_pool = np.concatenate((search_pool, similarity), axis=1)
        # sort the data in descending order
        search_pool = search_pool[search_pool[:, -1].argsort()][::-1]
        return search_pool[:self.top_n_results]

In [20]:
model_path = 'sentence-transformers/all-MiniLM-L6-v2'
obj = HardMineNegatives(
    model_path=model_path,
    search_max_threshold=65,
    search_min_threshold=50,
    search_limit=None,
    top_n_results=1)

/home/kperkins411/anaconda3/envs/p311/lib/python3.11/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [21]:
final_results = []
for row in tqdm(df.to_dict('records')):
    anchor = row['anchor']
    search_pool = df[df.anchor != anchor]
    search_pool.reset_index(drop=True, inplace=True)
    search_pool = search_pool.drop_duplicates()
    search_pool = search_pool.loc[:, ['positive']]
    search_pool = search_pool.to_numpy()
    print(f'Mining negatives for "{anchor}"')
    top_results = obj.get_hard_mined_negatives(anchor, search_pool)
    anchor = np.array([anchor]).reshape(-1, 1)
    anchor = np.repeat(anchor, repeats=len(top_results), axis=0)
    results_with_meta = np.concatenate((anchor, top_results), axis=1)
    final_results.extend(results_with_meta.tolist())

  0%|          | 0/3347 [00:00<?, ?it/s]

Mining negatives for "what safeguards are in place to protect the information obtained from third-party sources?"


  0%|          | 1/3347 [00:01<1:49:25,  1.96s/it]

Mining negatives for "is there a guarantee from the manufacturers regarding the conformity of the items to the mutually approved written standards for a certain duration?"


  0%|          | 2/3347 [00:03<1:45:06,  1.89s/it]

...

Mining negatives for "does section 9.02 require notice for termination?"


  9%|▊         | 288/3347 [08:36<1:34:19,  1.85s/it]

Mining negatives for "what constitutes a 'special price' sale under this agreement?"


  9%|▊         | 289/3347 [08:38<1:34:45,  1.86s/it]

Mining negatives for "can analytics providers access user data over time?"
